In [1]:
import numpy as np

class ReLU:
    def forward(self, x):
        return np.maximum(0, x)
    def backward(self, x):
        return (x > 0).astype(float)

class Tanh:
    def forward(self, x):
        return np.tanh(x)
    def backward(self, x):
        return 1- (np.tanh(x)**2)

class Sigmoid:
    def forward(self, x):
        return 1/(1+np.exp(-x))
    def backward(self, x):
        sigma = 1/(1+np.exp(-x))
        return sigma*(1-sigma)

class Linear:
    def forward(self, x):
        return x
    def backward(self, x):
        return np.ones_like(x)

class MSE:
    def forward_loss(self, x, y):
        return np.mean((y-x)**2)
    def backward_loss(self, x, y):
        return (-2*(y-x))/len(x)

class BCE:
    def forward_loss(self, x, y):
        ep = 1e-16
        return -np.mean((y * np.log(x+ep))+((1-y)*np.log(ep+1-x)))
    def backward_loss(self, x, y):
        ep = 1e-16
        return ((x-y)/((x+ep)*(1-x+ep)))/len(y)

class Dense:
    def __init__(self, input_size, output_size):
        self.w = np.random.randn(input_size, output_size) * 0.000001 #W (R^input x output) is our weight matrix.
        self.b = np.zeros((1, output_size)) #b (R^1 x output) is our bias vector.
    def forward(self, x):
        self.x = x
        y0 = (x @ self.w)+self.b
        return y0
    def backward(self, dl_by_dy0):
        m = len(self.x)
        self.dl_by_dW = (self.x.T @ dl_by_dy0) / m
        self.dl_by_db = np.sum(dl_by_dy0, axis=0, keepdims=True) / m
        dx = dl_by_dy0 @ self.W.T
        return dx

# Backward Pass of a Dense Layer

## Forward Equation

For a dense (fully connected) layer:

$$
Y_0 = XW + b
$$

where:

| Variable | Shape | Meaning |
|---|---|---|
| $X$ | $(m,d)$ | input matrix |
| $W$ | $(d,k)$ | weight matrix |
| $b$ | $(1,k)$ | bias vector |
| $Y_0$ | $(m,k)$ | output before activation |

---

# What Does $\frac{\partial L}{\partial Y_0}$ Mean?

$$
\frac{\partial L}{\partial Y_0}
$$

means:

> gradient of the loss $L$ with respect to the layer output $Y_0$

In code:

```python
dl_by_dy0
```

Shape: $(m,k)$

Each element:

$$
\left(\frac{\partial L}{\partial Y_0}\right)_{ij}
=
\frac{\partial L}{\partial y_{ij}}
$$

tells us:

> "If this output changes slightly, how much does the loss change?"

---

# Goal of Backpropagation

We want:

$$
\frac{\partial L}{\partial W},
\quad
\frac{\partial L}{\partial b},
\quad
\frac{\partial L}{\partial X}
$$

---

# 1. Weight Gradient

## Scalar Derivation

For one neuron:

$$
y_j = \sum_i x_i w_{ij} + b_j
$$

Derivative wrt weight:

$$
\frac{\partial y_j}{\partial w_{ij}} = x_i
$$

Apply chain rule:

$$
\frac{\partial L}{\partial w_{ij}}
=
\frac{\partial L}{\partial y_j}
\frac{\partial y_j}{\partial w_{ij}}
$$

Substitute:

$$
\frac{\partial L}{\partial w_{ij}}
=
\frac{\partial L}{\partial y_j} x_i
$$

---

## Matrix Form

For the whole batch:

| Matrix | Shape |
|---|---|
| $X^T$ | $(d,m)$ |
| $dY = \frac{\partial L}{\partial Y_0}$ | $(m,k)$ |

Matrix multiplication:

$$
(d,m) \;@\; (m,k) = (d,k)
$$

which matches weight shape.

Therefore:

$$
\boxed{
\frac{\partial L}{\partial W}
=
X^T \frac{\partial L}{\partial Y_0}
}
$$

If loss is averaged over batch:

$$
\boxed{
\frac{\partial L}{\partial W}
=
\frac{X^T \frac{\partial L}{\partial Y_0}}{m}
}
$$

Code:

```python
self.dl_by_dW = (self.x.T @ dl_by_dy0) / m
```

---

# 2. Bias Gradient

Since:

$$
Y_0 = XW + b
$$

for one neuron:

$$
\frac{\partial y_j}{\partial b_j} = 1
$$

Chain rule:

$$
\frac{\partial L}{\partial b_j}
=
\sum_i
\frac{\partial L}{\partial y_{ij}}
$$

because the same bias is shared across all samples.

So we sum over samples axis.

$$
\boxed{
\frac{\partial L}{\partial b}
=
\sum_{i=1}^{m}
\frac{\partial L}{\partial Y_0}
}
$$

Shape: $(m,k) \rightarrow (1,k)$

If using mean loss:

$$
\boxed{
\frac{\partial L}{\partial b}
=
\frac{
\sum \frac{\partial L}{\partial Y_0}
}{m}
}
$$

Code:

```python
self.dl_by_db = np.sum(dl_by_dy0, axis=0, keepdims=True) / m
```

In [ ]:
def singular_neuron(x, y):
    w = np.zeros(x.shape[1])